In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from datasets import Dataset

# 1. Load and prep your data
df = pd.read_csv("dataset.csv")
df = df.dropna(subset=["lyrics_cleaned", "parent_genre"])

le = LabelEncoder()
df["label"] = le.fit_transform(df["parent_genre"])
num_labels = len(le.classes_)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
train_df, val_df = train_test_split(
    train_df, test_size=0.1, stratify=train_df["label"], random_state=42
)

# 2. Tokenize
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["lyrics"],
        padding="max_length",
        truncation=True,
        max_length=512,
    )

train_ds = Dataset.from_pandas(train_df[["lyrics", "label"]]).map(tokenize, batched=True)
val_ds   = Dataset.from_pandas(val_df[["lyrics", "label"]]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[["lyrics", "label"]]).map(tokenize, batched=True)

# 3. Load the model with a fresh classification head
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=num_labels
)

# 4. Train
def compute_metrics(eval_pred):
    from sklearn.metrics import accuracy_score, f1_score
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

args = TrainingArguments(
    output_dir="./distilbert-genre",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
)

import torch
import torch.nn as nn
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from transformers import Trainer

# Compute the weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_df["label"]),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

# Custom Trainer that uses weighted loss
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device)
        )
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Use the custom trainer instead of Trainer
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

trainer.train()